# PaceAlgo ML — Notebook 06: Deep Analysis

**Zweck:** Bevor wir architektonisch entscheiden, ehrlich analysieren WAS der schwache PF 1.10 Edge eigentlich ist.

Vier kritische Fragen:

1. **SHAP:** Welche Features liefern tatsächlich die Vorhersagekraft? (LightGBM-gain ist eine Heuristik — SHAP zeigt die wahren Drivers.)
2. **Per-Regime:** Funktioniert das Modell nur in bestimmten Marktphasen? Vielleicht stark in Trend-Märkten, neutral in Range — dann filtern wir nach Regime.
3. **Top-N% Confidence:** Was passiert wenn wir nur die obersten 5/10/20% der Predictions handeln? Vielleicht ist der Edge in den allertopsten Confidence-Bars konzentriert.
4. **Meta-Labeling:** Statt ML als Primärsignal — ML als QUALITÄTS-FILTER auf rule-based Setups. Vielleicht ist da der echte Hebel.

**Output bestimmt nächsten Schritt:**
- Wenn Top-1% PF > 1.5 → es gibt Edge, brauchen nur engeren Threshold
- Wenn ein Regime PF > 1.4 → regime-gated trading möglich
- Wenn Meta-Filter rule-based PF von 1.x auf > 1.5 hebt → Hybrid-Architektur als V1
- Wenn nichts davon → Features ausbauen (Funding, OI, SMC) ist der einzige Weg

## 1. Environment + Load Existing Model

In [ ]:
import sys, os
from pathlib import Path
from datetime import datetime

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    MODELS_DIR = ARTIFACTS / 'models'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0:
        print('Restoring features from Drive backup...')
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_MODELS, ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS

sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Data:    {DATA_PROCESSED}')
print(f'Models:  {MODELS_DIR}')

In [ ]:
!pip install -q lightgbm shap 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import shap

from core.config import (
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES,
    DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
)
from core.train import (
    stack_symbols, get_feature_columns,
    walk_forward_split, binary_label_for_long,
    load_features_and_labels,
)
from core.analysis import (
    regime_buckets, performance_by_regime,
    confidence_percentile_sweep,
    rule_based_primary_signal,
    meta_labeling_evaluation,
)

R_VALUE = 1.5
ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS

# Load trained model from NB 05
model_path = MODELS_DIR / f'lgbm_R{R_VALUE}.txt'
meta_path = MODELS_DIR / f'lgbm_R{R_VALUE}_meta.json'
if not model_path.exists():
    print(f'ERROR: model not found at {model_path}')
    print('Run notebook 05 first to train and save the model.')
else:
    model = lgb.Booster(model_file=str(model_path))
    print(f'Model loaded: {model.num_trees()} trees')
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        feature_cols = meta['feature_cols']
        print(f'Features ({len(feature_cols)}): {feature_cols[:5]} ...')
        print(f'Threshold used during training: {meta["threshold"]}')
    else:
        feature_cols = None

## 2. Re-build Test Set + Predictions

In [ ]:
df = stack_symbols(
    DATA_PROCESSED,
    symbols=ALL_SYMBOLS,
    tfs=PRIMARY_TIMEFRAMES,
    R=R_VALUE,
    drop_holdout=DEV_HOLDOUT_SYMBOLS,
)
if feature_cols is None:
    feature_cols = get_feature_columns(df)
df_clean = df.dropna(subset=feature_cols)
_, val_df, test_df = walk_forward_split(df_clean, TRAIN_END, VAL_END)

X_test = test_df[feature_cols]
test_proba = model.predict(X_test)
print(f'Test set: {len(test_df):,} rows, predictions ready')
print(f'Probability range: {test_proba.min():.4f} - {test_proba.max():.4f}')
print(f'Probability mean:  {test_proba.mean():.4f}')

# Quick histogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(test_proba, bins=100, color='#3a7bd5', alpha=0.8)
ax.axvline(0.5, linestyle='--', color='red', label='naive threshold 0.5')
ax.axvline(test_proba.mean(), linestyle='--', color='orange', label=f'mean = {test_proba.mean():.3f}')
ax.set_xlabel('predicted probability')
ax.set_ylabel('count')
ax.set_title('Distribution of test-set predicted probabilities')
ax.legend()
plt.tight_layout()
plt.show()

## 3. SHAP Analysis — Was treibt das Modell wirklich?

LightGBM-Gain ist eine Heuristik. SHAP rechnet pro Bar aus wieviel jedes Feature zur Prediction beitrug — viel verlässlicher. 

Wir nehmen ein Sample von 10k Bars aus dem Test-Set (sonst wird's zu langsam).

In [ ]:
SAMPLE_N = 10_000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test), size=min(SAMPLE_N, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]

print(f'Computing SHAP values on {len(X_sample):,} test bars...')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
print(f'SHAP shape: {np.array(shap_values).shape}')

# Summary plot — top features by mean |SHAP value|
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols,
                    plot_type='bar', max_display=25, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm: shows DIRECTION of feature impact, not just magnitude
fig = plt.figure(figsize=(10, 9))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols,
                    max_display=20, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Mean |SHAP| as a clean table
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_table = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': mean_abs_shap,
    'shap_rank': np.argsort(np.argsort(-mean_abs_shap)) + 1,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
print('Features by mean absolute SHAP value:')
display(shap_table.round(5))

# Identify dead features (zero SHAP impact)
dead = shap_table[shap_table['mean_abs_shap'] < 1e-6]
print(f'\nDEAD features (no impact): {len(dead)} of {len(shap_table)}')
if len(dead) > 0:
    print(list(dead['feature']))

## 4. Per-Regime Performance

In [ ]:
test_with_regime = regime_buckets(test_df)

# Quick check: how does test data distribute across regimes?
print('Bars per regime_combo:')
print(test_with_regime['regime_combo'].value_counts().head(15))

print('\nBars per trend_regime:')
print(test_with_regime['trend_regime'].value_counts())

print('\nBars per vol_regime:')
print(test_with_regime['vol_regime'].value_counts())

In [ ]:
# Use chosen threshold from NB 05 (0.40 was best). Adjust if needed.
ANALYSIS_THRESHOLD = 0.40
print(f'Analyzing performance at threshold = {ANALYSIS_THRESHOLD}')

print('\n=== By trend regime ===')
by_trend = performance_by_regime(test_with_regime, test_proba, ANALYSIS_THRESHOLD,
                                   tp_R=R_VALUE, regime_col='trend_regime', min_n=500)
display(by_trend.round(4))

print('\n=== By volatility regime ===')
by_vol = performance_by_regime(test_with_regime, test_proba, ANALYSIS_THRESHOLD,
                                 tp_R=R_VALUE, regime_col='vol_regime', min_n=500)
display(by_vol.round(4))

print('\n=== By combo regime ===')
by_combo = performance_by_regime(test_with_regime, test_proba, ANALYSIS_THRESHOLD,
                                   tp_R=R_VALUE, regime_col='regime_combo', min_n=500)
display(by_combo.round(4))

## 5. Top-N% Confidence Sweep

Anstatt einen Threshold zu wählen, schauen wir uns die TOP N% der Predictions an. Wenn der Edge konzentriert ist in den allerhöchsten Confidences (z.B. Top 2%), sehen wir das hier sofort.

In [ ]:
sweep_pct = confidence_percentile_sweep(test_df['label'], test_proba, tp_R=R_VALUE)
print('Top-N% confidence sweep on TEST set:')
display(sweep_pct.round(4))

# Plot PF vs top_pct
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(sweep_pct['top_pct'], sweep_pct['profit_factor'], 'o-', color='#3a7bd5', label='Profit Factor')
ax1.axhline(1.0, linestyle='--', color='gray', alpha=0.5)
ax1.axhline(1.5, linestyle='--', color='green', alpha=0.5, label='PF=1.5 target')
ax1.set_xlabel('Top % of bars by confidence')
ax1.set_ylabel('Profit Factor')
ax1.set_xscale('log')
ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(sweep_pct['top_pct'], sweep_pct['win_rate'], 's-', color='orange', alpha=0.7, label='Win Rate')
ax2.set_ylabel('Win Rate', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.85))
ax1.set_title('Edge concentration: PF + WR by confidence percentile')
plt.tight_layout()
plt.show()

## 6. Meta-Labeling — ML als Quality Filter

Anstatt ML als Primärsignal zu nutzen, generieren wir Setups mit einem **einfachen Rule-Based Signal** (emuliert PaceAlgo v2.6 Style) und nutzen das ML-Modell nur als JA/NEIN-Filter auf diese Setups.

**Rule-Based Primary:**
- EMA Alignment +1 (Bull-Trend)
- ADX > 20
- Close innerhalb 1 ATR vom EMA20 (Pullback)
- RSI 30-70
- 1H HTF auch bullish

Wenn dieser Filter die richtigen 50-70% der primary signals aussortiert → wir haben einen klaren V1-Pfad.

In [ ]:
# Count primary signals first
primary = rule_based_primary_signal(test_df)
print(f'Rule-based primary fires on {primary.sum():,} of {len(primary):,} test bars ({primary.mean()*100:.2f}%)')

meta_results = meta_labeling_evaluation(
    test_df, test_proba, tp_R=R_VALUE,
    ml_thresholds=[0.0, 0.30, 0.35, 0.38, 0.40, 0.42, 0.44, 0.46],
)
print('\nMeta-labeling: rule-based primary -> ML quality filter')
display(meta_results.round(4))

# Highlight the lift
primary_only_pf = meta_results.iloc[0]['profit_factor']
print(f'\nPrimary alone PF:  {primary_only_pf:.3f}')
best_filtered = meta_results[meta_results['ml_threshold'].notna()]
if len(best_filtered) > 0:
    best_row = best_filtered.loc[best_filtered['profit_factor'].idxmax()]
    print(f'Best filtered PF:  {best_row["profit_factor"]:.3f} at ML threshold {best_row["ml_threshold"]}')
    print(f'  -> ML keeps {best_row["kept_pct"]*100:.1f}% of primary signals')
    print(f'  -> PF lift: {best_row["profit_factor"] - primary_only_pf:+.3f}')

## 7. Decision Matrix

In [ ]:
print('=' * 70)
print('ARCHITECTURE DECISION MATRIX')
print('=' * 70)

# 1. ML alone at chosen threshold
ml_alone_pf = sweep_pct[sweep_pct['top_pct'] == 10]['profit_factor'].values[0] if 10 in sweep_pct['top_pct'].values else None
print(f'\n1. ML alone (top 10% confidence): PF = {ml_alone_pf:.3f}' if ml_alone_pf else '\n1. ML alone — see sweep table')

# 2. Best single regime
if not by_combo.empty:
    best_regime = by_combo.iloc[0]
    print(f'2. Best single regime ({best_regime["regime"]}): PF = {best_regime["profit_factor"]:.3f}, n={int(best_regime["n"])}')

# 3. Top 1-2% confidence
top_1pct = sweep_pct[sweep_pct['top_pct'] == 1]
if len(top_1pct) > 0:
    print(f'3. Top 1% confidence: PF = {top_1pct.iloc[0]["profit_factor"]:.3f}, n={int(top_1pct.iloc[0]["n_selected"])}')

# 4. Meta-labeling
if len(meta_results) > 1:
    base_pf = meta_results.iloc[0]['profit_factor']
    filtered = meta_results[meta_results['ml_threshold'].notna()]
    if len(filtered) > 0:
        best_meta = filtered.loc[filtered['profit_factor'].idxmax()]
        print(f'4. Meta-labeling (rule + ML filter): primary {base_pf:.3f} -> filtered {best_meta["profit_factor"]:.3f}')
        print(f'   At ML threshold {best_meta["ml_threshold"]}, keeping {best_meta["kept_pct"]*100:.1f}% of signals')

print('\n' + '=' * 70)
print('Recommendation logic:')
print('=' * 70)
print('  If Top 1-2% PF > 1.5         -> ML works, just need tighter threshold')
print('  If single regime PF > 1.4     -> add regime gate (e.g. only trade in bull_low_vol_aligned)')
print('  If meta-PF > primary+0.2      -> Hybrid V1 (rule + ML filter)')
print('  If nothing reaches > 1.3      -> features are the bottleneck -> Funding/OI/SMC next')

---

Schick die Outputs von Section 3 (SHAP table), Section 4 (per-regime), Section 5 (top-N% + Plot), Section 6 (meta-labeling) und Section 7 (decision matrix). Daraus sehen wir genau wo wir stehen.